# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [2]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [6]:
# Task 1
# Keep the two requested room types; shared rooms are excluded.
room_types = ['Entire home/apt', 'Private room']

price_rooms = df_cap.loc[
    df_cap['room_type'].isin(room_types)
].copy()

# Calculate median prices.
medians = price_rooms.groupby('room_type')['price'].median()

room_colours = {
    'Entire home/apt': '#2E75B6',
    'Private room': '#70AD47'
}

# Create histogram.
fig = px.histogram(
    price_rooms,
    x='price',
    color='room_type',
    barmode='overlay',
    opacity=0.6,
    nbins=50,
    color_discrete_map=room_colours,
    category_orders={'room_type': room_types},
    labels={
        'price': 'Nightly price (£)',
        'count': 'Number of listings',
        'room_type': 'Room type'
    },
    height=560
)

# Add median lines without annotations.
fig.add_vline(
    x=medians['Entire home/apt'],
    line_dash='dash',
    line_color=room_colours['Entire home/apt'],
    line_width=2
)

fig.add_vline(
    x=medians['Private room'],
    line_dash='dash',
    line_color=room_colours['Private room'],
    line_width=2
)

# Place labels in fixed positions at opposite sides of the chart.
fig.add_annotation(
    x=0.02,
    y=0.98,
    xref='paper',
    yref='paper',
    text=f"Entire home median: £{medians['Entire home/apt']:.0f}",
    showarrow=False,
    xanchor='left',
    yanchor='top',
    font=dict(color=room_colours['Entire home/apt'], size=12),
    bgcolor='white',
    bordercolor=room_colours['Entire home/apt'],
    borderwidth=1,
    borderpad=6
)

fig.add_annotation(
    x=0.98,
    y=0.98,
    xref='paper',
    yref='paper',
    text=f"Private room median: £{medians['Private room']:.0f}",
    showarrow=False,
    xanchor='right',
    yanchor='top',
    font=dict(color=room_colours['Private room'], size=12),
    bgcolor='white',
    bordercolor=room_colours['Private room'],
    borderwidth=1,
    borderpad=6
)

# Improve layout.
fig.update_layout(
    title=dict(
        text=(
            f"Entire homes have a higher typical nightly price than "
            f"private rooms — prices above £{p95:.0f} "
            f"(95th percentile) excluded"
        ),
        x=0.5,
        xanchor='center'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(
        gridcolor='#EEEEEE',
        title='Number of listings'
    ),
    xaxis=dict(
        showgrid=False,
        title='Nightly price (£)'
    ),
    legend_title_text='Room type',
    margin=dict(l=60, r=60, t=120, b=50)
)

fig.show()

## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [10]:
# Task 2
# Use neighbourhood as the borough field in this dataset.
# Remove listings with zero or missing reviews per month.
activity = df.loc[df['reviews_per_month'].notna() & (df['reviews_per_month'] > 0)].copy()
 
# Compute median activity for each borough and order from lowest to highest.
borough_medians = activity.groupby('neighbourhood')['reviews_per_month'].median().sort_values()
borough_order = borough_medians.index.tolist()
top_two = borough_medians.sort_values(ascending=False).head(2).index.tolist()
 
print("Two most active boroughs by median reviews per month:")
print(borough_medians.loc[top_two].round(2))
 
# Use grey for the other boroughs and a meaningful highlight for the two most active.
fig = px.box(
    activity,
    x='reviews_per_month',
    y='neighbourhood',
    orientation='h',
    color='neighbourhood',
    category_orders={'neighbourhood': borough_order},
    points='outliers',
    color_discrete_sequence=['#DDDDDD'] * len(borough_order),
    labels={'reviews_per_month': 'Reviews per month', 'neighbourhood': 'Borough'},
    height=max(600, 22 * len(borough_order))
)
 
# Grey all boroughs first, then highlight the two most active.
for trace in fig.data:
    if trace.name in top_two:
        trace.marker.color = "#E80E2B"
        trace.line.color = "#B6E820"
    else:
        trace.marker.color = '#BBBBBB'
        trace.line.color = '#999999'
 
# Overall median reference line, annotated for context.
overall_median = activity['reviews_per_month'].median()
fig.add_vline(
    x=overall_median,
    line_dash='dash',
    line_color="#10139F",
    line_width=2,
    annotation_text=f'Overall median: {overall_median:.2f} reviews/month',
    annotation_position='top right',
    annotation_font_color="#77126A"
)
 
fig.update_layout(
    title=f"{top_two[0]} and {top_two[1]} have the highest median listing activity",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    xaxis=dict(gridcolor='#EEEEEE', title='Reviews per month'),
    yaxis=dict(showgrid=False, title='Borough'),
    showlegend=False,
    margin=dict(l=180, r=50, t=80, b=45)
)
fig.show()

Two most active boroughs by median reviews per month:
neighbourhood
Tower Hamlets    1.75
Hackney          1.66
Name: reviews_per_month, dtype: float64
